# Censored Data HAL Density Estimation

This notebook demonstrates HAL-based density estimation for **right-censored data** on [0,1].

## Problem Setup

For right-censored data we observe `(T, Δ)` where:
- `T = min(X, C)` — observed time (minimum of true event time and censoring time)
- `Δ = I(X ≤ C)` — event indicator (1 if observed, 0 if censored)

## Methods Covered

| Method | Description |
|--------|-------------|
| **IPCW-HAL-MLE** | Inverse probability of censoring weighted HAL on uncensored observations |
| **EMStage** | Standalone EM refinement using any initial working model |
| **EMIPCWEstimator** | Bundled IPCW initialization + EM refinement |

## Hyperparameter Tuning Options

| Tuner | What It Tunes | Use Case |
|-------|---------------|----------|
| `CensoredOptunaHyperparameterTuner` (IPCW) | `basis_order`, `norm_constraint` | Fast IPCW-only tuning |
| `CensoredOptunaHyperparameterTuner` (EM) | + `m_step_norm_multiplier` [0.5–1.0] | Joint EM tuning |
| `TwoStageCensoredTuner` | Stage 1: IPCW params → Stage 2: EM multiplier | Efficient two-stage |
| `EMStageTuner` | `m_step_norm_multiplier` only | Pre-fitted initial model |

## Notebook Outline

1. **Data Simulation** — Generate right-censored truncated normal data
2. **Kaplan-Meier** — Estimate censoring survival function
3. **IPCW Baseline** — Fit initial weighted HAL estimate
4. **EMStage Refinement** — Refine IPCW estimate via EM
5. **EMIPCWEstimator** — Bundled IPCW + EM workflow
6. **Hyperparameter Tuning** — CV-based optimization
7. **Comparison** — Evaluate all methods


In [ ]:
# Imports
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import truncnorm

from haldensity.censoring import (
    # Utilities
    KaplanMeier,
    compute_ipcw_weights,
    # Estimators
    WeightedCVXPYEstimator,
    EMStage,
    EMIPCWEstimator,
    # Tuners
    CensoredOptunaHyperparameterTuner,
    TwoStageCensoredTuner,
    EMStageTuner,
    # Metrics
    kl_divergence,
    incomplete_loglik,
)

print("✓ Imports successful")


## 1. Data Simulation

Generate right-censored data from a **truncated normal** distribution on [0,1]:
- True event times: `X ~ TruncatedNormal(μ=0.5, σ=0.1)`
- Censoring times: `C ~ Uniform(0, 1)`
- Observed: `T = min(X, C)` and `Δ = I(X ≤ C)`


In [ ]:
# Simulation parameters
np.random.seed(42)
rng = np.random.default_rng(42)

n_samples = 1000
mean, std = 0.5, 0.1
lower, upper = 0.0, 1.0

# Generate truncated normal event times
a, b = (lower - mean) / std, (upper - mean) / std
X_true = truncnorm.rvs(a, b, loc=mean, scale=std, size=n_samples, random_state=rng)

# Generate uniform censoring times
C = rng.uniform(lower, upper, size=n_samples)

# Observed data
T = np.minimum(X_true, C)
Delta = (X_true <= C).astype(int)

# Store in DataFrame
data = pd.DataFrame({"T": T, "Delta": Delta})

print(f"Sample size: {n_samples}")
print(f"Censoring rate: {(1 - Delta.mean()):.1%}")
print(f"Observed events: {Delta.sum()}")
print(f"Censored observations: {(1 - Delta).sum()}")


In [ ]:
# Visualize data
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Observed data histogram
axes[0].hist(data.loc[data['Delta'] == 1, 'T'], bins=30, alpha=0.6, label='Events', density=True)
axes[0].hist(data.loc[data['Delta'] == 0, 'T'], bins=30, alpha=0.6, label='Censored', density=True)
axes[0].set_xlabel('Observed Time T')
axes[0].set_ylabel('Density')
axes[0].set_title('Observed Data Distribution')
axes[0].legend()
axes[0].grid(alpha=0.3)

# True density
grid = np.linspace(0, 1, 500)
true_density = truncnorm.pdf(grid, a, b, loc=mean, scale=std)
axes[1].plot(grid, true_density, 'k-', linewidth=2, label='True Density')
axes[1].hist(X_true, bins=30, alpha=0.3, density=True, label='True Events (unobserved)')
axes[1].set_xlabel('X')
axes[1].set_ylabel('Density')
axes[1].set_title('True Event Time Distribution')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()


## 2. Kaplan-Meier Censoring Survival

Estimate the **censoring survival function** `S_c(t) = P(C > t)` using Kaplan-Meier. This is needed for IPCW weights.


In [ ]:
# Fit Kaplan-Meier for censoring survival
km = KaplanMeier()
km.fit(data, time_col="T", delta_col="Delta")

# Get step function for plotting
km_times, km_surv = km.stepwise_survival_()

print(f"KM estimated at {len(km_times)} unique time points")
print(f"S_c(0.0) = {km.predict(0.0):.3f}")
print(f"S_c(0.5) = {km.predict(0.5):.3f}")
print(f"S_c(1.0) = {km.predict(1.0):.3f}")

# Plot
plt.figure(figsize=(8, 5))
plt.step(km_times, km_surv, where='post', linewidth=2, label='KM $S_c(t)$')
plt.xlabel('Time t')
plt.ylabel('Censoring Survival Probability')
plt.title('Kaplan-Meier Estimate of Censoring Survival')
plt.grid(alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
# Compute IPCW weights
T_vals = np.asarray(data["T"].values, dtype=float)
Delta_vals = np.asarray(data["Delta"].values, dtype=int)

def S_c_predict(t):
    """Predict censoring survival - returns np.ndarray."""
    return np.atleast_1d(km.predict(t))

ipcw_weights = compute_ipcw_weights(T=T_vals, Delta=Delta_vals, S_c_predict=S_c_predict)

print(f"IPCW weights: min={ipcw_weights.min():.2f}, max={ipcw_weights.max():.2f}, mean={ipcw_weights.mean():.2f}")
print(f"Non-zero weights (events): {(ipcw_weights > 0).sum()}")


## 3. IPCW-HAL-MLE (Initial Estimate)

Fit a **weighted HAL density estimator** using only uncensored observations with IPCW weights.

This serves as:
- A standalone baseline estimate
- The **initial working model** for EMStage refinement


In [ ]:
# Fit IPCW-weighted HAL estimator
uncensored_mask = Delta_vals == 1
ipcw_data = pd.DataFrame({"W1": T_vals[uncensored_mask]})
ipcw_weights_unc = ipcw_weights[uncensored_mask]

ipcw_estimator = WeightedCVXPYEstimator(
    norm_constraint=10,
    basis_order=0,
    solver="ECOS",
)
ipcw_estimator.fit(ipcw_data, sample_weights=ipcw_weights_unc)

# Results
ipcw_results = ipcw_estimator.get_results()
print(f"IPCW-HAL knots selected: {len(ipcw_results['grid_points_hal_selected'])}")

# Evaluate
eval_grid = np.linspace(0, 1, 500)
ipcw_density = ipcw_estimator.get_density_at_points(eval_grid)
ipcw_ll = incomplete_loglik(ipcw_estimator, data, time_col="T", delta_col="Delta")
ipcw_kl = kl_divergence(
    true_pdf_fn=lambda x: truncnorm.pdf(x, a, b, loc=mean, scale=std),
    grid=eval_grid,
    est_density=ipcw_density,
)

print(f"Log-likelihood: {ipcw_ll:.4f}")
print(f"KL divergence: {ipcw_kl:.6f}")


In [ ]:
# Plot IPCW estimate
plt.figure(figsize=(10, 6))
plt.plot(grid, true_density, 'k-', linewidth=2, label='True Density', alpha=0.8)
plt.plot(eval_grid, ipcw_density, 'b-', linewidth=2, label=f'IPCW-HAL (KL={ipcw_kl:.4f})', alpha=0.7)
plt.xlabel('x')
plt.ylabel('Density f(x)')
plt.title('IPCW-HAL-MLE vs True Density')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()


## 4. EMStage Refinement (Standalone)

Use **`EMStage.run()`** to refine the IPCW estimate via EM iterations with multiple imputation.

Key insight: `EMStage` takes **any initial estimator** as input, giving you full control over the workflow.


In [ ]:
# Configure and run EMStage
em_stage = EMStage(
    m_imputations=30,       # Imputations per censored observation
    max_em_iter=20,         # Maximum EM iterations
    em_tol=1e-4,            # Convergence tolerance
    norm_constraint=8,    # L1 norm constraint for M-step
    n_grid_points=200,
    m_step_solver="ECOS",
    verbose=False,
    rng_seed=42,
)

# Run EMStage with IPCW as initial working model
em_result = em_stage.run(
    initial_estimator=ipcw_estimator,
    data=data,
    S_c_predict=S_c_predict,
)

em_estimator = em_result.final_estimator
print(f"\n{'='*50}")
print(f"EMStage Results:")
print(f"  EM iterations: {em_result.em_iterations}")
print(f"  Converged: {em_result.em_converged}")


In [ ]:
# Evaluate EMStage results
em_density = em_estimator.get_density_at_points(eval_grid)
em_ll = incomplete_loglik(em_estimator, data, time_col="T", delta_col="Delta")
em_kl = kl_divergence(
    true_pdf_fn=lambda x: truncnorm.pdf(x, a, b, loc=mean, scale=std),
    grid=eval_grid,
    est_density=em_density,
)

print(f"EMStage Log-likelihood: {em_ll:.4f}")
print(f"EMStage KL divergence: {em_kl:.6f}")
print(f"\nImprovement over IPCW:")
print(f"  LL change: {em_ll - ipcw_ll:+.4f}")
print(f"  KL change: {em_kl - ipcw_kl:+.6f} (negative = better)")


In [ ]:
# Plot: IPCW vs EMStage vs True
plt.figure(figsize=(10, 6))
plt.plot(grid, true_density, 'k-', linewidth=2.5, label='True Density', alpha=0.8)
plt.plot(eval_grid, ipcw_density, 'b--', linewidth=2, label=f'IPCW (KL={ipcw_kl:.4f})', alpha=0.6)
plt.plot(eval_grid, em_density, 'r-', linewidth=2, label=f'EMStage (KL={em_kl:.4f})', alpha=0.7)
plt.xlabel('x')
plt.ylabel('Density f(x)')
plt.title('IPCW vs EMStage-Refined')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()


## 5. EMIPCWEstimator (Bundled)

EMIPCWEstimator bundles IPCW initialization and EM refinement into a single convenient estimator.


In [ ]:
# Fit bundled EMIPCWEstimator
bundled_estimator = EMIPCWEstimator(
    norm_constraint=50,
    m_step_norm_constraint=35,
    m_imputations=30,
    max_em_iter=10,
    em_tol=1e-4,
    basis_order=1,
    init_solver="ECOS",
    m_step_solver="ECOS",
    rng_seed=42,
    verbose=False,
)
bundled_estimator.fit(data)

# Evaluate
bundled_results = bundled_estimator.get_results()
bundled_density = bundled_estimator.get_density_at_points(eval_grid)
bundled_ll = incomplete_loglik(bundled_estimator, data, time_col="T", delta_col="Delta")
bundled_kl = kl_divergence(
    true_pdf_fn=lambda x: truncnorm.pdf(x, a, b, loc=mean, scale=std),
    grid=eval_grid,
    est_density=bundled_density,
)

print(f"\nEMIPCWEstimator:")
print(f"  EM iterations: {bundled_results['em_iterations']}")
print(f"  Log-likelihood: {bundled_ll:.4f}")
print(f"  KL divergence: {bundled_kl:.6f}")


In [ ]:
# Plot: Bundled EMIPCWEstimator vs True
plt.figure(figsize=(10, 6))
plt.plot(eval_grid, truncnorm.pdf(eval_grid, a, b, loc=mean, scale=std), 'k-', linewidth=2.5, label='True Density', alpha=0.8)
plt.plot(eval_grid, bundled_density, 'g-', linewidth=2, label='EMIPCWEstimator (Bundled)', alpha=0.7)
plt.xlabel('x')
plt.ylabel('Density f(x)')
plt.title('Bundled EMIPCWEstimator vs True')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 6. Hyperparameter Tuning

Three tuning approaches with different speed/flexibility tradeoffs:

| Approach | What It Tunes | When to Use |
|----------|--------------|-------------|
| **IPCW-only Tuner** | `basis_order`, `norm_constraint` | Fast baseline, no EM |
| **TwoStageCensoredTuner** | Stage 1: IPCW → Stage 2: EM multiplier | Efficient full workflow |
| **EMStageTuner** | `m_step_norm_multiplier` only | Pre-fitted IPCW model |


### 6a. IPCW-only Tuning (Fast Baseline)

Tune only `basis_order` and `norm_constraint` using IPCW — no EM iterations.


In [ ]:
# IPCW-only tuner (no EM)
ipcw_tuner = CensoredOptunaHyperparameterTuner(
    estimator_name="WeightedCVXPYEstimator",
    data=data,
    cv_folds=3,
    random_state=42,
    n_grid_points=200,
    param_overrides={
        "norm_constraint": {"low": 10.0, "high": 500.0, "log": True},
        "basis_order": [0, 1, 2],
    },
    silent=False,
)

start_time = time.time()
ipcw_result = ipcw_tuner.optimize(n_trials=50)
print(f"\nOptimization time: {time.time() - start_time:.1f}s")
print(f"Best params: {ipcw_result['best_params']}")
print(f"Best LL: {ipcw_result['best_metric_value']:.4f}")


In [ ]:
# Fit best IPCW model and evaluate
best_ipcw = ipcw_tuner.fit_best_model()
ipcw_tuned_density = best_ipcw.get_density_at_points(eval_grid)
ipcw_tuned_ll = incomplete_loglik(best_ipcw, data, time_col="T", delta_col="Delta")
ipcw_tuned_kl = kl_divergence(
    true_pdf_fn=lambda x: truncnorm.pdf(x, a, b, loc=mean, scale=std),
    grid=eval_grid,
    est_density=ipcw_tuned_density,
)
print(f"CV-Tuned IPCW: LL={ipcw_tuned_ll:.4f}, KL={ipcw_tuned_kl:.6f}")

# Plot
plt.figure(figsize=(10, 6))
plt.plot(grid, true_density, 'k-', linewidth=2.5, label='True Density', alpha=0.8)
plt.plot(eval_grid, ipcw_density, 'b--', linewidth=2, label=f'IPCW (KL={ipcw_kl:.4f})', alpha=0.6)
plt.plot(eval_grid, ipcw_tuned_density, 'g-', linewidth=2, label=f'CV-IPCW (KL={ipcw_tuned_kl:.4f})', alpha=0.7)
plt.xlabel('x')
plt.ylabel('Density f(x)')
plt.title('IPCW vs CV-Tuned IPCW')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()


### 6b. TwoStageCensoredTuner

Stage 1: Fast IPCW-only tuning for `basis_order` and `norm_constraint`  
Stage 2: EM tuning for `m_step_norm_multiplier` only (with Stage 1 params fixed)


In [ ]:
# Two-stage tuner
two_stage_tuner = TwoStageCensoredTuner(
    data=data,
    cv_folds=3,
    random_state=42,
    stage1_param_ranges={
        "norm_constraint": {"low": 0.1, "high": 500.0, "log": True},
        "basis_order": [0, 1, 2],
    },
    stage2_param_ranges={
        "m_step_norm_multiplier": {"low": 0.5, "high": 1.0, "log": True},
    },
    em_defaults={"m_imputations": 20, "max_em_iter": 10},
    silent=False,
)

start_time = time.time()
best_params = two_stage_tuner.optimize(n_trials_stage1=30, n_trials_stage2=10)
print(f"\nTotal optimization time: {time.time() - start_time:.1f}s")


In [ ]:
# Fit best two-stage model
tuned_result = two_stage_tuner.fit_best_model()
tuned_estimator = tuned_result.final_estimator
tuned_density = tuned_estimator.get_density_at_points(eval_grid)
tuned_ll = incomplete_loglik(tuned_estimator, data, time_col="T", delta_col="Delta")
tuned_kl = kl_divergence(
    true_pdf_fn=lambda x: truncnorm.pdf(x, a, b, loc=mean, scale=std),
    grid=eval_grid,
    est_density=tuned_density,
)
print(f"Two-Stage Tuned: LL={tuned_ll:.4f}, KL={tuned_kl:.6f}")

# Plot
plt.figure(figsize=(10, 6))
plt.plot(grid, true_density, 'k-', linewidth=2.5, label='True Density', alpha=0.8)
plt.plot(eval_grid, ipcw_density, 'b--', linewidth=2, label=f'IPCW (KL={ipcw_kl:.4f})', alpha=0.6)
plt.plot(eval_grid, tuned_density, 'r-', linewidth=2, label=f'TwoStage (KL={tuned_kl:.4f})', alpha=0.7)
plt.xlabel('x')
plt.ylabel('Density f(x)')
plt.title('TwoStageCensoredTuner Result')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()


### 6c. EMStageTuner (Pre-fitted Initial)

Use this when you already have a fitted IPCW estimator and want to tune only the EM refinement.


In [ ]:
# EMStageTuner with pre-fitted IPCW model
em_tuner = EMStageTuner(
    data=data,
    initial_estimator=best_ipcw,  # Use CV-tuned IPCW from above
    S_c_predict=S_c_predict,
    cv_folds=3,
    param_ranges={"m_step_norm_multiplier": {"low": 0.5, "high": 1.0, "log": True}},
    em_defaults={"m_imputations": 20, "max_em_iter": 10},
    silent=True,
)

em_best_params = em_tuner.optimize(n_trials=20)


In [ ]:
em_best_params

In [ ]:
# Fit best EMStageTuner model
em_tuner_result = em_tuner.fit_best_model()
em_tuner_estimator = em_tuner_result.final_estimator
em_tuner_density = em_tuner_estimator.get_density_at_points(eval_grid)
em_tuner_ll = incomplete_loglik(em_tuner_estimator, data, time_col="T", delta_col="Delta")
em_tuner_kl = kl_divergence(
    true_pdf_fn=lambda x: truncnorm.pdf(x, a, b, loc=mean, scale=std),
    grid=eval_grid,
    est_density=em_tuner_density,
)
print(f"EMStageTuner: LL={em_tuner_ll:.4f}, KL={em_tuner_kl:.6f}")

# Plot
plt.figure(figsize=(10, 6))
plt.plot(grid, true_density, 'k-', linewidth=2.5, label='True Density', alpha=0.8)
plt.plot(eval_grid, ipcw_tuned_density, 'b--', linewidth=2, label=f'CV-IPCW (KL={ipcw_tuned_kl:.4f})', alpha=0.6)
plt.plot(eval_grid, em_tuner_density, 'm-', linewidth=2, label=f'EMStageTuner (KL={em_tuner_kl:.4f})', alpha=0.7)
plt.xlabel('x')
plt.ylabel('Density f(x)')
plt.title('CV-IPCW vs EMStageTuner Refinement')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()


## 7. Summary Comparison

Compare all methods: baseline IPCW, manual EMStage, bundled EMIPCWEstimator, and all tuned variants.


In [ ]:
# Summary table
print("=" * 70)
print("SUMMARY: All Methods Comparison")
print("=" * 70)

results_summary = pd.DataFrame({
    "Method": [
        "IPCW (baseline)",
        "EMStage (manual)",
        "EMIPCWEstimator",
        "CV-IPCW",
        "TwoStageTuner",
        "EMStageTuner",
    ],
    "Log-Likelihood": [ipcw_ll, em_ll, bundled_ll, ipcw_tuned_ll, tuned_ll, em_tuner_ll],
    "KL Divergence": [ipcw_kl, em_kl, bundled_kl, ipcw_tuned_kl, tuned_kl, em_tuner_kl],
})
print(results_summary.to_string(index=False))
print("=" * 70)


In [ ]:
# Final comparison plot
plt.figure(figsize=(14, 8))
plt.plot(grid, true_density, 'k-', linewidth=3, label='True Density', alpha=0.9)
plt.plot(eval_grid, ipcw_density, 'b--', linewidth=2, label=f'IPCW baseline (KL={ipcw_kl:.4f})', alpha=0.5)
plt.plot(eval_grid, em_density, 'c-', linewidth=2, label=f'EMStage manual (KL={em_kl:.4f})', alpha=0.6)
plt.plot(eval_grid, ipcw_tuned_density, 'orange', linewidth=2, label=f'CV-IPCW (KL={ipcw_tuned_kl:.4f})', alpha=0.6)
plt.plot(eval_grid, tuned_density, 'r-', linewidth=2, label=f'TwoStageTuner (KL={tuned_kl:.4f})', alpha=0.7)
plt.plot(eval_grid, em_tuner_density, 'm-', linewidth=2.5, label=f'EMStageTuner (KL={em_tuner_kl:.4f})', alpha=0.8)
plt.xlabel('x', fontsize=12)
plt.ylabel('Density f(x)', fontsize=12)
plt.title('All Density Estimation Methods', fontsize=14)
plt.legend(fontsize=10, loc='upper right')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()


## Key Takeaways

### Estimators

| Estimator | Description | Use When |
|-----------|-------------|----------|
| `WeightedCVXPYEstimator` | IPCW-weighted HAL | Fast baseline |
| `EMStage` | Standalone EM refinement | Full control over initial model |
| `EMIPCWEstimator` | Bundled IPCW + EM | Convenience |

### Tuners

| Tuner | Tunes | Best For |
|-------|-------|----------|
| `CensoredOptunaHyperparameterTuner` (IPCW) | `basis_order`, `norm_constraint` | Fast IPCW-only tuning |
| `TwoStageCensoredTuner` | Stage 1: IPCW, Stage 2: EM multiplier | Full pipeline optimization |
| `EMStageTuner` | `m_step_norm_multiplier` only | When IPCW is already tuned |

### Workflow Recommendation

1. **Start with IPCW-only tuning** to find good `basis_order` and `norm_constraint`
2. **Add EMStage refinement** if needed for better fit
3. **Use TwoStageCensoredTuner** for automated end-to-end optimization
